In [1]:
import os
from datetime import datetime
import numpy as np
from evaluate import load
import random
from sklearn.metrics import classification_report
from pandas import DataFrame as DF

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
os.environ["DGLBACKEND"] = "pytorch"
import dgl
import dgl.data
from dgl.nn.pytorch import SAGEConv

In [3]:
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
dgl.seed(4056)
torch.manual_seed(4056)
torch.cuda.manual_seed(4056)
torch.backends.cudnn.benchmarks = False
torch.backends.cudnn.deterministic = True
np.random.seed(4056)
random.seed(4056)

In [4]:
device = torch.device('cuda:0')

### Utils

In [5]:
from load_mpdd_data import load_mpdd_data
from load_ldc_data import load_ldc_chinese_data

In [13]:
mpdd_graphs = load_mpdd_data()

Translations:   0%|          | 0/11643 [00:00<?, ?it/s]

186248
1809 96944 87495
2951 384 807


Dialogues:   0%|          | 0/4142 [00:00<?, ?it/s]

In [19]:
g = mpdd_graphs['1']
# display(g)
for n in g['effects']:
    print(n[0])

1. Disrespectful language:
- Observed effect: It can create tension and animosity between 左母 (Mrs. Zuo) and 左正鵬 (Zho Zpeng).

2. Opposition towards 左正鵬's (Zho Zpeng) relationship:
- Observed effect: It causes disagreement and arguments between 左母 (Mrs. Zuo) and 左父 (Mr. Zuo), highlighting a lack of unity within the family.



In [ ]:
mpdd_graphs = load_mpdd_data()

In [ ]:
fpath = '/home/rpujari/gilbreth_scratch/scratch_ml/DARPA/'
mpdd_path = '/home/rpujari/gilbreth_scratch/scratch_ml/DARPA/mpdd/'
save_path = '/home/rpujari/gilbreth_scratch/scratch_ml/DARPA/ta2snapshot_saved_parameters/'

In [ ]:
# en_model_name = 'distilroberta-base'
en_model_name = 'roberta-base'
zh_model_name = 'hfl/chinese-roberta-wwm-ext'

### Create Graphs

In [ ]:
plutchik =  ['anger', 'fear', 'sadness', 'disgust', 'surprise', 'anticipation', 'trust', 'joy', 'neutral']

mpdd_emotion_map = {
    'fear': 'fear',
    'angry': 'anger',
    'disgust': 'disgust',
    'sadness': 'sadness',
    'happiness': 'joy',
    'surprise': 'surprise',
    'neutral': 'neutral'
}

cped_emotion_map = {
    'grateful': 'trust',
    'neutral': 'neutral',
    'astonished': 'surprise',
    'sadness': 'sadness',
    'fear': 'fear',
    'worried': 'anticipation',
    'anger': 'anger',
    'depress': 'sadness',
    'disgust': 'disgust',
    'happy': 'joy',
    'relaxed': 'joy',
    'positive-other': 'joy',
    'negative-other': 'sadness'
}

ldc_norm_ids = {
        101: 'apology',
        102: 'criticism',
        103: 'greeting',
        104: 'request',
        105: 'persuasion',
        106: 'thanking',
        107: 'leave',
        108: 'admiration',
        109: 'negotiation',
        110: 'refusal',
        150: 'other'
}

In [ ]:
# Make graph structures from graph dictionaries
def add_edge(src_name, dest_name, adj_list, name2id, bidirectional=False):
    src_id = name2id[src_name]
    dest_id = name2id[dest_name]
    if src_id not in adj_list:
        adj_list[src_id] = []
    adj_list[src_id].append(dest_id)
    if bidirectional:
        if dest_id not in adj_list:
            adj_list[dest_id] = []
        adj_list[dest_id].append(src_id)

In [ ]:
#Function to create graph structures from graph dictionaries
def create_graph(graph_dict, d_id, prefix='', only_relevant=False):
    id2names = {}
    name2id = {}
    node_docs = {}
    adj_list = {}
    idx = 0

    #create dialogue node
    id2names[idx] = prefix + '-dialogue-' + d_id
    name2id[prefix + '-dialogue-' + d_id] = idx
    node_docs[idx] = graph_dict['dialogue']
    idx += 1

    #create summary node
    if 'summary' in graph_dict:
        id2names[idx] = prefix + '-summary-' + d_id
        name2id[prefix + '-summary-' + d_id] = idx
        node_docs[idx] = graph_dict['summary']
        idx += 1

    #create metadata node
    if 'metadata' in graph_dict:
        id2names[idx] = prefix + '-metadata-' + d_id
        name2id[prefix + '-metadata-' + d_id] = idx
        node_docs[idx] = graph_dict['metadata']
        idx += 1

    #create field nodes if they don't exist already
    if 'field' in graph_dict:
        for field in graph_dict['field']:
            field_node_name = prefix + '-field-' + field[0]
            if field_node_name not in name2id:
                id2names[idx] = field_node_name
                name2id[field_node_name] = idx
                #first occurring bprop mode used for all occurrences
                node_docs[idx] = field
                idx += 1

    #create turn nodes
    for i, turn in enumerate(graph_dict['turns']):
        id2names[idx] = prefix + '-turn-' + d_id + '-' + str(i)
        node_docs[idx] = turn
        name2id[prefix + '-turn-' + d_id + '-' + str(i)] = idx
        idx += 1

        #create position nodes if they don't exist already
        if 'positions' in graph_dict:
            for position in graph_dict['positions'][i]:
                position_node_name = prefix + '-position-' + position[0]
                if position_node_name not in name2id:
                    id2names[idx] = position_node_name
                    name2id[position_node_name] = idx
                    #first occurring bprop mode used for all occurrences
                    node_docs[idx] = position
                    idx += 1

        #create relationship nodes if they don't exist already
        if 'relationships' in graph_dict:
            for relationship in graph_dict['relationships'][i]:
                relationship_node_name = prefix + '-relationship-' + relationship[0]
                if relationship_node_name not in name2id:
                    id2names[idx] = relationship_node_name
                    name2id[relationship_node_name] = idx
                    #first occurring bprop mode used for all occurrences
                    node_docs[idx] = relationship
                    idx += 1

    #create theme nodes if they don't exist already
    if 'norm_themes' in graph_dict:
        for i, theme in enumerate(graph_dict['norm_themes']):
            if theme[0] != 'Unknown':
                theme_name = theme[0].strip().split('\n')[0].split(':')[1].strip().lower().replace(' ', '_')
                theme_node_name = prefix + '-theme-' + theme_name
                if theme_node_name not in name2id:
                    id2names[idx] = theme_node_name
                    name2id[theme_node_name] = idx
                    #first occurring bprop mode used for all occurrences
                    node_docs[idx] = theme
                    idx += 1

    #create norm description nodes
    if 'norms' in graph_dict:
        for i, norm in enumerate(graph_dict['norms']):
            relevance = graph_dict['norm_relevance'][i].lower() if (only_relevant and 'norm_relevance' in graph_dict) else 'relevant'
            if relevance == 'relevant':
                id2names[idx] = prefix + '-norm-' + d_id + '-' + str(i)
                node_docs[idx] = norm
                name2id[prefix + '-norm-' + d_id + '-' + str(i)] = idx
                idx += 1

    #create norm description nodes
    if 'symbolic_data' in graph_dict:
        for i, symbol in enumerate(graph_dict['symbolic_data']):
            if symbol[0].strip() != '':
                id2names[idx] = prefix + '-symbol-' + d_id + '-' + str(i)
                node_docs[idx] = symbol
                name2id[prefix + '-symbol-' + d_id + '-' + str(i)] = idx
                idx += 1

    #create violation description nodes
    if 'violations' in graph_dict:
        for i, violation in enumerate(graph_dict['violations']):
            id2names[idx] = prefix + '-violation-' + d_id + '-' + str(i)
            node_docs[idx] = violation
            name2id[prefix + '-violation-' + d_id + '-' + str(i)] = idx
            idx += 1

    #create effect description nodes
    if 'effects' in graph_dict:
        for i, effect in enumerate(graph_dict['effects']):
            id2names[idx] = prefix + '-effect-' + d_id + '-' + str(i)
            node_docs[idx] = effect
            name2id[prefix + '-effect-' + d_id + '-' + str(i)] = idx
            idx += 1

    #add edge from summary to dialogue, both ways
    if 'summary' in graph_dict:
        add_edge(prefix + '-summary-' + d_id, prefix + '-dialogue-' + d_id, adj_list, name2id, bidirectional=True)
        #add edge from metadata to summary, both ways
        if 'metadata' in graph_dict:
            add_edge(prefix + '-metadata-' + d_id, prefix + '-summary-' + d_id, adj_list, name2id, bidirectional=True)

    #add edge from metadata to dialogue, both ways
    if 'metadata' in graph_dict:
        add_edge(prefix + '-metadata-' + d_id, prefix + '-dialogue-' + d_id, adj_list, name2id, bidirectional=True)

    #add edge from field to dialogue & summary
    if 'field' in graph_dict:
        for field in graph_dict['field']:
            field_node_name = prefix + '-field-' + field[0]
            add_edge(field_node_name, prefix + '-dialogue-' + d_id, adj_list, name2id, bidirectional=False)
            if 'summary' in graph_dict:
                add_edge(field_node_name, prefix + '-summary-' + d_id, adj_list, name2id, bidirectional=False)

    for i, turn in enumerate(graph_dict['turns']):
        turn_node_name = prefix + '-turn-' + d_id + '-' + str(i)

        #add edge from turn to dialogue & summary, both ways
        add_edge(turn_node_name, prefix + '-dialogue-' + d_id, adj_list, name2id, bidirectional=True)
        if 'summary' in graph_dict:
            add_edge(turn_node_name, prefix + '-summary-' + d_id, adj_list, name2id, bidirectional=True)

        #add edge from positions to turn
        if 'positions' in graph_dict:
            for position in graph_dict['positions'][i]:
                position_node_name = prefix + '-position-' + position[0]
                add_edge(position_node_name, turn_node_name, adj_list, name2id, bidirectional=False)

        #add edge from relationships to turn
        if 'relationships' in graph_dict:
            for relationship in graph_dict['relationships'][i]:
                relationship_node_name = prefix + '-relationship-' + relationship[0]
                add_edge(relationship_node_name, turn_node_name, adj_list, name2id, bidirectional=False)

        #add edge from norm descriptions to all turns, both ways
        if 'norms' in graph_dict:
            for i, norm in enumerate(graph_dict['norms']):
                relevance = graph_dict['norm_relevance'][i].lower() if (only_relevant and 'norm_relevance' in graph_dict) else 'relevant'
                if relevance == 'relevant':
                    norm_node_name = prefix + '-norm-' + d_id + '-' + str(i)
                    add_edge(norm_node_name, turn_node_name, adj_list, name2id, bidirectional=True)

        #add edge from violation descriptions to all turns, both ways
        if 'violations' in graph_dict:
            for i, violation in enumerate(graph_dict['violations']):
                violation_node_name = prefix + '-violation-' + d_id + '-' + str(i)
                add_edge(violation_node_name, turn_node_name, adj_list, name2id, bidirectional=True)

        #add edge from effect descriptions to all turns, both ways
        if 'effects' in graph_dict:
            for i, effect in enumerate(graph_dict['effects']):
                effect_node_name = prefix + '-effect-' + d_id + '-' + str(i)
                add_edge(effect_node_name, turn_node_name, adj_list, name2id, bidirectional=True)

    #add edge from norm descriptions to dialogue & summary, both ways
    if 'norms' in graph_dict:
        for i, norm in enumerate(graph_dict['norms']):
            relevance = graph_dict['norm_relevance'][i].lower() if (only_relevant and 'norm_relevance' in graph_dict) else 'relevant'
            if relevance == 'relevant':
                add_edge(prefix + '-norm-' + d_id + '-' + str(i), prefix + '-dialogue-' + d_id, adj_list, name2id, bidirectional=True)
                if 'summary' in graph_dict:
                    add_edge(prefix + '-norm-' + d_id + '-' + str(i), prefix + '-summary-' + d_id, adj_list, name2id, bidirectional=True)

    #add edge from norm descriptions to theme, both ways
    if 'norm_themes' in graph_dict:
        for i, theme in enumerate(graph_dict['norm_themes']):
            relevance = graph_dict['norm_relevance'][i].lower() if (only_relevant and 'norm_relevance' in graph_dict) else 'relevant'
            if relevance == 'relevant' and theme[0] != 'Unknown':
                theme_name = theme[0].strip().split('\n')[0].split(':')[1].strip().lower().replace(' ', '_')
                theme_node_name = prefix + '-theme-' + theme_name
                add_edge(prefix + '-norm-' + d_id + '-' + str(i), theme_node_name, adj_list, name2id, bidirectional=True)

    #add edge from norm descriptions to theme, both ways
    if 'symbolic_data' in graph_dict:
        for i, symbol in enumerate(graph_dict['symbolic_data']):
            relevance = graph_dict['norm_relevance'][i].lower() if (only_relevant and 'norm_relevance' in graph_dict) else 'relevant'
            if relevance == 'relevant' and symbol[0].strip() != '':
                symbol_node_name = prefix + '-symbol-' + d_id + '-' + str(i)
                add_edge(prefix + '-norm-' + d_id + '-' + str(i), symbol_node_name, adj_list, name2id, bidirectional=True)

    #add edge from violation descriptions to dialogue & summary, both ways
    if 'violations' in graph_dict:
        for i, violation in enumerate(graph_dict['violations']):
            add_edge(prefix + '-violation-' + d_id + '-' + str(i), prefix + '-dialogue-' + d_id, adj_list, name2id, bidirectional=True)
            if 'summary' in graph_dict:
                add_edge(prefix + '-violation-' + d_id + '-' + str(i), prefix + '-summary-' + d_id, adj_list, name2id, bidirectional=True)

    #add edge from effect descriptions to dialogue & summary, both ways
    if 'effects' in graph_dict:
        for i, effect in enumerate(graph_dict['effects']):
            add_edge(prefix + '-effect-' + d_id + '-' + str(i), prefix + '-dialogue-' + d_id, adj_list, name2id, bidirectional=True)
            if 'summary' in graph_dict:
                add_edge(prefix + '-effect-' + d_id + '-' + str(i), prefix + '-summary-' + d_id, adj_list, name2id, bidirectional=True)
            
    for nid in node_docs:
        assert(type(node_docs[nid]) == tuple and len(node_docs[nid]) == 2)
    
    return (graph_dict, id2names, name2id, node_docs, adj_list)

In [ ]:
tokenizers = {
    'en': AutoTokenizer.from_pretrained(en_model_name),
    'en-bprop': AutoTokenizer.from_pretrained(en_model_name),
    'zh': AutoTokenizer.from_pretrained(zh_model_name),
    'zh-bprop': AutoTokenizer.from_pretrained(zh_model_name),
}

languages = sorted(list(tokenizers.keys()))
print(languages)

# Convert graph structures into DGL graphs
def convert_graph_to_input(G, node_ids, labels, data_types):
    gd, id2n, n2id, nd, adj_list = G
    num_nodes = len(id2n)
    src = []
    dest = []
    train_mask = [False] * num_nodes
    valid_mask = [False] * num_nodes
    test_mask = [False] * num_nodes
    node_labels = [-1] * num_nodes
    node_languages = [0] * num_nodes
    
    for i, l, dt in zip(node_ids, labels, data_types):
        if dt == 'train':
            train_mask[i] = True
        elif dt == 'valid':
            valid_mask[i] = True
        elif dt == 'test':
            test_mask[i] = True
        node_labels[i] = l
        
    node_docs = {}
    node_lang_ids = {}
    #accumulate docs and node_ids for each language
    for i in range(num_nodes):
        doc, lang = nd[i]
        if lang not in node_docs:
            node_docs[lang] = [doc]
            node_lang_ids[lang] = [i]
        else:
            node_docs[lang].append(doc)
            node_lang_ids[lang].append(i)
        
        node_languages[i] = languages.index(lang)
        
        #initialize adj list with edges
        if i in adj_list:
            for j in adj_list[i]:
                src.append(i)
                dest.append(j)
                
    combined_node_input_ids = torch.LongTensor(num_nodes, 500)
    combined_node_attn_mask = torch.LongTensor(num_nodes, 500)
    
    for lang in node_docs:
        #tokenize documents of that language
        node_docs_toks = tokenizers[lang].batch_encode_plus(node_docs[lang], return_tensors="pt", padding='max_length', max_length=500, truncation=True)
        
        #place tokens in the correct location in the combined token_outputs
        i = 0
        for id_ in node_lang_ids[lang]:
            combined_node_input_ids[id_, :] = node_docs_toks['input_ids'][i, :]
            combined_node_attn_mask[id_, :] = node_docs_toks['attention_mask'][i, :]
            i += 1
        assert(i == node_docs_toks['input_ids'].size(0))
    
        
    ret_g = dgl.graph((torch.LongTensor(src), torch.LongTensor(dest)), num_nodes=num_nodes)
    ret_g.ndata['input_ids'] = combined_node_input_ids
    ret_g.ndata['attention_mask'] = combined_node_attn_mask
    ret_g.ndata['label'] = torch.LongTensor(node_labels)
    ret_g.ndata['language'] = torch.LongTensor(node_languages)
    ret_g.ndata['train_mask'] = torch.BoolTensor(train_mask)
    ret_g.ndata['valid_mask'] = torch.BoolTensor(valid_mask)
    ret_g.ndata['test_mask'] = torch.BoolTensor(test_mask)
    ret_g = dgl.add_self_loop(ret_g)
    return ret_g

In [ ]:
# task_category = 'norm_categories'
# default_label = 'other'

task_category = 'emotions'
default_label = 'neutral'

task_labels = set()
for d_id in mpdd_graphs:
    for pos in mpdd_graphs[d_id][task_category]:
        task_labels.add(pos)
task_label_list = sorted(list(task_labels))
print(len(task_label_list))
print(task_label_list)

In [ ]:
t1 = datetime.now()
random.seed(4056)

train_graphs = {}
val_graphs = {}
test_graphs = {}

lc = [0] * len(task_label_list)
random.seed(4056)
c = 0

for d_id in mpdd_graphs:
    g_dict = mpdd_graphs[d_id]
    g = create_graph(g_dict, d_id)
    node_ids = []
    node_labels = []
    node_data_types = []
    id2n = g[1]
    for id_ in id2n:
        name = id2n[id_]
        if 'turn' in name:
            tnum = int(name.split('-')[-1])
            node_ids.append(id_)
            node_labels.append(task_label_list.index(g_dict[task_category][tnum]))
            node_data_types.append(g_dict['split'])
            
    if len(node_labels) > 0:
        g1 = convert_graph_to_input(g, node_ids, node_labels, node_data_types)
        if g_dict['split'] == 'train':
            train_graphs[d_id] = g1
            for i in range(len(task_label_list)):
                lc[i] += node_labels.count(i)
        elif g_dict['split'] == 'valid':
            val_graphs[d_id] = g1
        elif g_dict['split'] == 'test':
            test_graphs[d_id] = g1
            c += int(sum(g1.ndata['test_mask']))
        
print(len(train_graphs), len(val_graphs), len(test_graphs), c)
print(lc)
t2 = datetime.now()
print(t2-t1)

In [ ]:
def batchify_graphs(graph_dict, max_node_num=150):
    all_dids = []
    for d_id in graph_dict:
        all_dids.append(d_id)
    sorted_dids = sorted(all_dids, key=lambda x:graph_dict[d_id].num_nodes())

    all_batches = []
    b = 0
    w = 0
    while b < len(sorted_dids):
        b_size = 0
        batch = []
        batch_dids = []
        while b < len(sorted_dids) and (b_size + graph_dict[sorted_dids[b]].num_nodes()) > max_node_num:
            w += 1
            b += 1
        # sorting apiori means next graphs are only going to be bigger
        while b < len(sorted_dids) and (b_size + graph_dict[sorted_dids[b]].num_nodes()) <= max_node_num:
            if graph_dict[sorted_dids[b]].num_nodes() <= max_node_num:
                batch.append(graph_dict[sorted_dids[b]])
                batch_dids.append(sorted_dids[b])
                b_size += graph_dict[sorted_dids[b]].num_nodes()
            else:
                w += 1
            b += 1
        # print(b, b_size, graph_dict[sorted_dids[b]].num_nodes())
        if len(batch) > 0:
            batch_graph = dgl.batch(batch)
            all_batches.append((batch_graph, batch_dids))
    print('Discarded graphs: ', w)
    return all_batches

In [ ]:
train_batches = batchify_graphs(train_graphs, max_node_num=100)
val_batches = batchify_graphs(val_graphs, max_node_num=300)
test_batches = batchify_graphs(test_graphs, max_node_num=300)

print(len(train_batches), len(val_batches), len(test_batches))

In [ ]:
class GraphSAGE(nn.Module):
    def __init__(self, in_feats, h_feats, num_classes, encoder_bprop=True):
        super(GraphSAGE, self).__init__()
        self.en_encoder = AutoModel.from_pretrained(en_model_name)
        self.en_encoder_bprop = AutoModel.from_pretrained(en_model_name)
        self.zh_encoder = AutoModel.from_pretrained(zh_model_name)
        self.zh_encoder_bprop = AutoModel.from_pretrained(zh_model_name)
        self.conv1 = SAGEConv(in_feats, h_feats, aggregator_type='mean')
        self.conv2 = SAGEConv(h_feats, num_classes, aggregator_type='mean')
        
        self.conv_mid_1 = SAGEConv(h_feats, h_feats, aggregator_type='mean')
        self.conv_mid_2 = SAGEConv(h_feats, h_feats, aggregator_type='mean')
        self.conv_mid_3 = SAGEConv(h_feats, h_feats, aggregator_type='mean')
        self.conv_mid_4 = SAGEConv(h_feats, h_feats, aggregator_type='mean')

        for p in self.en_encoder.parameters():
            p.requires_grad = False
        for p in self.zh_encoder.parameters():
            p.requires_grad = False
        if encoder_bprop == False:
            for p in self.en_encoder_bprop.parameters():
                p.requires_grad = False
            for p in self.zh_encoder_bprop.parameters():
                p.requires_grad = False
                
        # Initialize the GraphSAGE layers randomly
        self._initialize_weights()

    def _initialize_weights(self):
        """Randomly initialize the weights of the DGL SAGEConv layers."""
        for layer in [self.conv1, self.conv2, self.conv_mid_1, self.conv_mid_2, self.conv_mid_3, self.conv_mid_4]:
            nn.init.xavier_uniform_(layer.fc_neigh.weight)  # Xavier initialization for the weight matrix
            nn.init.xavier_uniform_(layer.fc_self.weight)  # Xavier initialization for the weight matrix
            if layer.fc_self.bias is not None:
                nn.init.zeros_(layer.fc_self.bias)  # Set bias to zeros
                
    def forward(self, g, input_ids, attention_mask, langs):
        num_nodes = langs.size(0)
        
        sorted_ids = sorted(list(range(num_nodes)), key=lambda x: langs[x])
        remap_ids = [sorted_ids.index(i) for i in range(num_nodes)]
        
        sorted_input_ids = input_ids[sorted_ids, :].to(input_ids.device)
        sorted_attention_mask = attention_mask[sorted_ids, :].to(attention_mask.device)
        
        boundaries = []
        data_langs = []
        prev_lang = -1
        for i in range(num_nodes):
            if langs[sorted_ids[i]] != prev_lang:
                boundaries.append(i)
                prev_lang = langs[sorted_ids[i]]
                data_langs.append(langs[sorted_ids[i]])
        boundaries.append(num_nodes)
        
        in_feats = []
        i = 0
        for b, e in zip(boundaries[:-1], boundaries[1:]):
            if int(data_langs[i]) == 0:
                in_feat = self.en_encoder(input_ids=sorted_input_ids[b:e, :], attention_mask=sorted_attention_mask[b:e, :]).last_hidden_state[:, 0, :].contiguous()
            elif int(data_langs[i]) == 1:
                in_feat = self.en_encoder_bprop(input_ids=sorted_input_ids[b:e, :], attention_mask=sorted_attention_mask[b:e, :]).last_hidden_state[:, 0, :].contiguous()
            elif int(data_langs[i]) == 2:
                in_feat = self.zh_encoder(input_ids=sorted_input_ids[b:e, :], attention_mask=sorted_attention_mask[b:e, :]).last_hidden_state[:, 0, :].contiguous()
            elif int(data_langs[i]) == 3:
                in_feat = self.zh_encoder_bprop(input_ids=sorted_input_ids[b:e, :], attention_mask=sorted_attention_mask[b:e, :]).last_hidden_state[:, 0, :].contiguous()
            in_feats.append(in_feat)
            i += 1
        
        in_feats = torch.cat(in_feats, dim=0).contiguous()
        in_feats = in_feats[remap_ids, :].contiguous()
        
        # Sampling neighbors
        g = g.local_var()
        
        h = self.conv1(g, in_feats)
        h = F.relu(h)
        
        h = self.conv2(g, h)
        
        return h.squeeze(1)

In [ ]:
def inference(model, test_data, mask_name='test_mask'):
    t1 = datetime.now()

    val_corr = 0
    val_tot = 0
    v_lc_e = [0] * len(task_label_list) 
    # model.eval()
    metric = load('f1', average='macro')
    preds = []
    golds = []
    for batch, batch_dids in test_data:
        torch.cuda.empty_cache()
        g_val = batch
        g_val = g_val.to(device)
        input_ids = g_val.ndata['input_ids']
        attn_mask = g_val.ndata['attention_mask']
        val_mask = g_val.ndata[mask_name]
        labels = g_val.ndata['label']
        langs = g_val.ndata['language']
        # Forward
        with torch.no_grad():
            logits = model(g_val, input_ids, attn_mask, langs)

            # Compute prediction
            pred = logits.argmax(1)
            # print(pred)
            for i in range(len(task_label_list)):
                v_lc_e[i] += list(pred[val_mask]).count(i)

            val_corr += (pred[val_mask] == labels[val_mask]).cpu().float().sum()
            preds.append(pred[val_mask])
            golds.append(labels[val_mask])
            metric.add_batch(predictions=pred[val_mask], references=labels[val_mask])
            val_tot += val_mask.cpu().float().sum()

    all_pred = torch.cat(preds, dim=0).cpu().data.numpy()
    all_gold = torch.cat(golds, dim=0).cpu().data.numpy()
    
    print(all_pred.shape, all_gold.shape)
    
    report = classification_report(all_gold, all_pred, labels = list(range(len(task_label_list))), target_names=task_label_list, zero_division=0, output_dict=True)
    report_df = DF(report).transpose()
    
    # Compute accuracy on validation
    if 'accuracy' in report:
        val_acc = report['accuracy']
    elif 'micro avg' in report:
        val_acc = report['micro avg']['f1-score']
    else:
        val_acc = val_corr / val_tot
    
    val_f1 = report['weighted avg']['f1-score']

    t2 = datetime.now()
    print(val_acc, val_f1, t2-t1)
    display(report_df)
    # print(report)
    
    return val_acc, val_f1, report

In [ ]:
def train(model, train_data, val_data, num_epochs=10, model_path='model.pt', lr=1e-5):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    best_val_acc = 0
    best_val_f1 = 0
    best_test_acc = 0
    
    t1 = datetime.now()
    
    val_acc, val_f1, report = inference(model, val_data, 'valid_mask')
    # Compute accuracy on validation
    if 'accuracy' in report:
        val_acc = report['accuracy']
    elif 'micro avg' in report:
        val_acc = report['micro avg']['f1-score']
    else:
        val_acc = val_corr / val_tot

    val_f1 = report['weighted avg']['f1-score']

    # Save the best validation accuracy and the corresponding test accuracy.
    best_val_f1 = val_f1
    best_val_acc = val_acc
    
    # model.train()
    for e in range(num_epochs):
        random.shuffle(train_data)
        train_loss = 0
        train_tot = 0
        i = 0
        for batch, batch_dids in train_data:
            g = batch
            g = g.to(device)
            input_ids = g.ndata['input_ids']
            attn_mask = g.ndata['attention_mask']
            train_mask = g.ndata['train_mask']
            labels = g.ndata['label']
            langs = g.ndata['language']

            # Forward
            logits = model(g, input_ids, attn_mask, langs)
            # Compute prediction
            pred = logits.argmax(1)
            # print(pred[train_mask])
            # Compute loss
            # Note that you should only compute the losses of the nodes in the training set.
            loss = F.cross_entropy(logits[train_mask], labels[train_mask], weight=(1.0 / torch.LongTensor(lc)).to(device))
            train_loss += loss.cpu().data
            train_tot += train_mask.cpu().float().sum()

            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            torch.cuda.empty_cache()
            i += 1
            
        train_loss /= train_tot
        val_acc, val_f1, report = inference(model, val_data, 'valid_mask')
        # Compute accuracy on validation
        if 'accuracy' in report:
            val_acc = report['accuracy']
        elif 'micro avg' in report:
            val_acc = report['micro avg']['f1-score']
        else:
            val_acc = val_corr / val_tot

        val_f1 = report['weighted avg']['f1-score']

        # Save the best validation accuracy and the corresponding test accuracy.
        if best_val_f1 < val_f1:
            best_val_f1 = val_f1
            best_val_acc = val_acc
            model = model.to('cpu')
            torch.save(model.state_dict(), save_path + model_path)
            torch.cuda.empty_cache()
            model = model.to(device)

        t2 = datetime.now()
        print(
            f"In epoch {e}, val acc: {val_acc:.3f}, val f1: {val_f1:.3f} (best {best_val_f1:.3f}, {t2-t1})"
        )

In [ ]:
torch.cuda.empty_cache()
model1 = GraphSAGE(768, 100, len(task_label_list), encoder_bprop=True)
# model1.load_state_dict(torch.load(save_path + 'schema_roberta_mpdd_norm/model.pt'))
device = torch.device('cuda:0')
model1 = model1.to(device)

In [ ]:
train(model1, train_batches, val_batches, num_epochs=50,\
      model_path='schema_roberta_mpdd_emotion/distilbert_graphsage_500_6L_full_bprop.pt', lr=1e-5)

In [ ]:
torch.cuda.empty_cache()
device = torch.device('cuda:0')
model3 = GraphSAGE(768, 100, len(task_label_list))
state_dict = torch.load(save_path + 'schema_roberta_mpdd_emotion/distilbert_graphsage_500_6L_full_bprop.pt')
# encoder_state_dict = {k[8:]: v for k, v in state_dict.items() if k.startswith('encoder')}
# model3.en_encoder.load_state_dict(encoder_state_dict)
model3.load_state_dict(state_dict)
model3 = model3.to(device)

ret = inference(model3, test_batches)